# Variational Autoencoder (VAE) für Synthetische Daten
Reconstructed Notebook

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import os
import pickle

# ------------------------- Reproduzierbarkeit -------------------------
torch.manual_seed(42)
np.random.seed(42)

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [2]:

# ------------------------- Daten laden -------------------------
from pathlib import Path
current_dir = Path.cwd()
notebooks_root = current_dir.parents[2]
if not notebooks_root.name == "Jupyter Notebooks":
    for p in current_dir.parents:
        if p.name == "Jupyter Notebooks":
            notebooks_root = p
            break

preprocessing_dir = notebooks_root / "3_Machine-Learning" / "3.1_Preprocessing" / "Preprocessing"
print(f"Preprocessing Verzeichnis: {preprocessing_dir}")

if not preprocessing_dir.exists():
    # Fallback auf relativ, falls absolut fehlschlägt
    preprocessing_dir = Path("../../../3_Machine-Learning/3.1_Preprocessing/Preprocessing")

if not preprocessing_dir.exists():
     raise FileNotFoundError(f"Verzeichnis existiert nicht: {preprocessing_dir}")

all_subdirs = [d for d in preprocessing_dir.iterdir() if d.is_dir()]
if not all_subdirs:
    raise FileNotFoundError("Keine Preprocessing-Ordner gefunden.")

latest_subdir = max(all_subdirs, key=os.path.getmtime)
data_path = latest_subdir / 'Preprocessed_SOM_Ready.csv'

print(f"Lade Daten aus: {data_path}")

try:
    df = pd.read_csv(data_path, low_memory=False)
    print(f"Datensatzgröße: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"Fehler beim Laden: {e}")
    raise e


# ------------------------- Gesteinsart-Analyse -------------------------
# ------------------------- Anforderung: Keine unbekannten Gesteinsarten trainieren. -------------------------
if 'rock_type' in df.columns:
    initial_len = len(df)
    df['rock_type'] = df['rock_type'].astype(str)
    
# ------------------------- Filterung -------------------------
    df = df[df['rock_type'] != 'Unknown'].copy()
    df = df[df['rock_type'] != 'nan'].copy()
    
    filtered_len = len(df)
    print(f"Filtered 'Unknown' Rock Types: {initial_len} -> {filtered_len} rows (Removed {initial_len - filtered_len})")
else:
    print("Warning: 'rock_type' column not found for filtering.")


Preprocessing Verzeichnis: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing
Lade Daten aus: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.1_Preprocessing\Preprocessing\2026-01-08_00-24-31\Preprocessed_SOM_Ready.csv


Datensatzgröße: (94264, 24)


,WGS84_latitude,WGS84_longitude,Database_number,temperature_in_c,rock_type,redox_potential_in_mV_gauss,total_dissolved_solids_in_mmol/L_log_gauss,O2_in_mmol/L_log_gauss,Na_in_mmol/L_log_gauss,Mg_in_mmol/L_log_gauss,...,Li_in_mmol/L_log_gauss,K_in_mmol/L_log_gauss,Sr_in_umol/L_log_gauss,NH4_in_umol/L_log_gauss,Fe_in_mmol/L_log_gauss,Mn_in_mmol/L_log_gauss,F_in_umol/L_log_gauss,NO3_in_mmol/L_log_gauss,H2SiO3_in_umol/L_log_gauss,HS_in_mmol/L_log_gauss
0,46.19680,8.54160,5,25.7,Andesite,NaN,-2.407685,NaN,-2.002380,-2.696511,...,-1.767903,-2.225823,NaN,NaN,NaN,NaN,0.590949,NaN,NaN,NaN
1,46.24352,9.72534,5,37.9,Basalt,NaN,-2.153160,NaN,-1.875864,-2.823144,...,-1.634747,-1.625274,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,45.76189,6.96890,5,22.3,Andesite,NaN,-1.933503,NaN,-2.035320,-0.746430,...,-1.404533,-1.674186,NaN,NaN,NaN,NaN,-0.926176,NaN,NaN,NaN
3,46.26274,8.10919,5,29.2,Andesite,NaN,-1.847320,NaN,-3.276940,-0.702315,...,NaN,-1.837975,NaN,NaN,NaN,NaN,-0.926176,NaN,NaN,NaN
4,46.26274,8.10919,5,38.6,Andesite,NaN,-1.732235,NaN,-2.709462,-0.653377,...,NaN,-1.775299,NaN,NaN,NaN,NaN,-0.926176,NaN,NaN,NaN


Filtered 'Unknown' Rock Types: 94264 -> 24332 rows (Removed 69932)


In [3]:

# ------------------------- Daten Preparation & Feature Engineering -------------------------

# 1. Temperatur-Behandlung (Vor Entfernenping schützen)
if 'temperature_in_c' in df.columns:
    df['temperature_in_c'] = pd.to_numeric(df['temperature_in_c'], errors='coerce')
    temp_median = df['temperature_in_c'].median()
    if pd.isna(temp_median): temp_median = df['temperature_in_c'].mean() 
    if pd.isna(temp_median): temp_median = 0 
    df['temperature_in_c'] = df['temperature_in_c'].fillna(temp_median)
    print(f"Temperature filled with: {temp_median}")
else:
    print("WARNING: 'temperature_in_c' not found!")

# ------------------------- Gesteinsart-Analyse -------------------------
if 'rock_type' in df.columns:
    df['rock_type'] = df['rock_type'].fillna('Unknown')
    rock_dummies = pd.get_dummies(df['rock_type'], prefix='rock', dtype=float)
    df_clean = pd.concat([df, rock_dummies], axis=1)
    df_clean.drop(columns=['rock_type'], inplace=True)
    rock_cols = rock_dummies.columns.tolist()
    print(f"Added {len(rock_cols)} rock type columns.")
else:
    print("WARNING: 'rock_type' not found!")
    df_clean = df.copy()
    rock_cols = []

# ------------------------- Filterung -------------------------
drop_threshold = 80.0 # Stricter threshold for VAE quality
missing_percentage = df_clean.isnull().mean() * 100
cols_to_drop = missing_percentage[missing_percentage > drop_threshold].index.tolist()
protected_cols = ['temperature_in_c'] + rock_cols
cols_to_drop = [c for c in cols_to_drop if c not in protected_cols]

if cols_to_drop:
    print(f"Dropping: {cols_to_drop}")
    df_clean = df_clean.drop(columns=cols_to_drop)
else:
    print("No columns dropped.")

# ------------------------- 4. Exclude Metadata -------------------------
metadata_cols = ['Database_number', 'database_name', 'Database_name', 'Date', 'Day', 'Month', 'Year', 'WGS84_latitude', 'WGS84_longitude']
cols_to_exclude = [c for c in metadata_cols if c in df_clean.columns]
if cols_to_exclude:
    df_train = df_clean.drop(columns=cols_to_exclude)
else:
    df_train = df_clean

# ------------------------- 5. Numeric Prep -------------------------
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
for c in numeric_cols:
    df_train[c] = pd.to_numeric(df_train[c], errors='coerce')
data_values = df_train[numeric_cols].values

# ------------------------- 6. Imputation & Scaling -------------------------
print("Imputing & Scaling...")
imputer = SimpleImputer(strategy='mean')
data_imputed = imputer.fit_transform(data_values)
# --- SCIENTIFIC FIX: GAUSSIAN ANAMORPHOSIS ---
# VAEs assume N(0,1) latent space. Heavy-tailed geochemical data violates this.
# ------------------------- Verteilungsanalyse -------------------------
scaler = QuantileTransformer(output_distribution='normal', n_quantiles=max(min(len(df_clean)//2, 1000), 10), random_state=42)
data_scaled = scaler.fit_transform(data_imputed)

# ------------------------- Feature-Namen explizit definieren -------------------------
feature_names = numeric_cols
print(f"Final Features ({len(feature_names)}): {feature_names}")

# 7. Persistierung (Metadaten speichern, um NameFehler zu vermeiden)
metadata = {
    'feature_names': feature_names,
    'rock_cols': rock_cols,
    'scaler': scaler,
    'imputer': imputer
}
with open('vae_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
print("Metadaten gespeichert unter vae_metadata.pkl")

# ------------------------- 8. Tensors -------------------------
tensor_data = torch.FloatTensor(data_scaled)
batch_size = 64
dataset = TensorDataset(tensor_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


Temperature filled with: 24.45
Added 30 rock type columns.
Dropping: ['redox_potential_in_mV_gauss', 'O2_in_mmol/L_log_gauss', 'Li_in_mmol/L_log_gauss', 'Sr_in_umol/L_log_gauss', 'NH4_in_umol/L_log_gauss', 'Fe_in_mmol/L_log_gauss', 'Mn_in_mmol/L_log_gauss', 'F_in_umol/L_log_gauss', 'NO3_in_mmol/L_log_gauss', 'H2SiO3_in_umol/L_log_gauss', 'HS_in_mmol/L_log_gauss']
Imputing & Scaling...


Final Features (39): ['temperature_in_c', 'total_dissolved_solids_in_mmol/L_log_gauss', 'Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L_log_gauss', 'K_in_mmol/L_log_gauss', 'rock_Andesite', 'rock_Anhydrite', 'rock_Basalt', 'rock_Breccia', 'rock_Chat', 'rock_Chert', 'rock_Claystone', 'rock_Coal', 'rock_Conglomerate', 'rock_Dolerite', 'rock_Dolomite', 'rock_Gneiss', 'rock_Granite', 'rock_Graywacke', 'rock_Greenschist', 'rock_Gypsum', 'rock_Limestone', 'rock_Marble', 'rock_Marl', 'rock_Metamorphic Rock', 'rock_Phyllite', 'rock_Quartzite', 'rock_Rhyolite', 'rock_Salt', 'rock_Sand and Gravel', 'rock_Sandstone', 'rock_Schist', 'rock_Shale', 'rock_Siltstone', 'rock_Slate']
Metadaten gespeichert unter vae_metadata.pkl


In [ ]:

# ------------------------- WISSENSCHAFTLICHE OPTIMIERUNG: BETA-VAE -------------------------
# Erhöhte Kapazität und Beta-Skalierung für bessere Entflechtung des Latent Space
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, latent_dim=16): # Optimierte Dimensionen
        super(VAE, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
        self.activation = nn.ReLU()

    def encode(self, x):
        h1 = self.activation(self.fc1(x))
        return self.fc2_mu(h1), self.fc2_logvar(h1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = self.activation(self.fc3(z))
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# Hier definiert für Verfügbarkeit
def loss_function(recon_x, x, mu, logvar, beta=1.0):
    MSE = nn.MSELoss(reduction='sum')(recon_x, x)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    # Beta-VAE: KLD mit Beta gewichten
    return MSE + beta * KLD, MSE, KLD

# ------------------------- Initialisierung -------------------------
input_dim = data_scaled.shape[1]
latent_dim = 16 # Optimized
model = VAE(input_dim, hidden_dim=256, latent_dim=latent_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
print("Optimiertes Beta-VAE Modell initialisiert.")


In [ ]:

# ------------------------- OPTIMIERTES TRAINING (KL Annealing) -------------------------
epochs = 150
MAX_BETA = 0.05
WARMUP_EPOCHS = 50

print(f"Starte optimiertes VAE-Training (Epochs={epochs}, Max Beta={MAX_BETA})...")
model.train()
history_loss = []
history_mse = []
history_kld = []

for epoch in range(epochs):
    # KL-Annealing Zeitplan
    if epoch < WARMUP_EPOCHS:
        beta = (epoch / WARMUP_EPOCHS) * MAX_BETA
    else:
        beta = MAX_BETA
        
    train_loss = 0
    train_mse = 0
    train_kld = 0
    
    for batch in dataloader:
        data = batch[0]
        optimizer.zero_grad()
        recon, mu, logvar = model(data)
        
        # Nutze Beta-Verlust
        loss, mse, kld = loss_function(recon, data, mu, logvar, beta=beta)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        train_mse += mse.item()
        train_kld += kld.item()
    
    avg_loss = train_loss / len(dataloader.dataset)
    avg_mse = train_mse / len(dataloader.dataset)
    avg_kld = train_kld / len(dataloader.dataset)
    
    history_loss.append(avg_loss)
    history_mse.append(avg_mse)
    history_kld.append(avg_kld)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss {avg_loss:.4f} | MSE: {avg_mse:.4f} | KLD: {avg_kld:.4f} | Beta: {beta:.5f}")

# Historie plotten
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(history_loss, label='Total Loss')
ax[0].set_title('Trainingsverlust')
ax[0].legend()
ax[1].plot(history_mse, label='MSE')
ax[1].plot(history_kld, label='KLD')
ax[1].set_title('Zerlegung')
ax[1].legend()
plt.show()


In [6]:
model.eval()
num_samples = len(df_clean)
print(f"Generating {num_samples} synthetic samples...")
with torch.no_grad():
    # Sample from standard normal (Gaussian Anamorphosis expectation)
    z = torch.randn(num_samples, latent_dim)
    recon_batch = model.decode(z)
    
# ------------------------- Verteilungsanalyse -------------------------
    synthetic_data = scaler.inverse_transform(recon_batch.numpy())
    
    # --- SAFETY: Handle NaNs or Infs ---
    if np.isnan(synthetic_data).any() or np.isinf(synthetic_data).any():
        print("WARNING: NaNs or Infs detected in synthetic data! Handling them...")
        synthetic_data = np.nan_to_num(synthetic_data, nan=0.0, posinf=999999, neginf=-999999)

    df_synthetic = pd.DataFrame(synthetic_data, columns=feature_names)
    
# ------------------------- Gesteinsart-Analyse -------------------------
    # Hinweis: rock_cols were excluded from VAE usually? Or included?
# ------------------------- Feature-Auswahl -------------------------
    # We should threshold them if they are one-hot
    for col in df_synthetic.columns:
        if col.startswith('rock_'):
            df_synthetic[col] = df_synthetic[col].apply(lambda x: 1 if x > 0.5 else 0)

print(f"Synthetic Data Created: {df_synthetic.shape}")
display(df_synthetic.head())


Generating 24332 synthetic samples...


Synthetic Data Created: (24332, 39)


,temperature_in_c,total_dissolved_solids_in_mmol/L_log_gauss,Na_in_mmol/L_log_gauss,Mg_in_mmol/L_log_gauss,Ca_in_mmol/L_log_gauss,Cl_in_mmol/L_log_gauss,SO4_in_mmol/L_log_gauss,HCO3_in_mmol/L_log_gauss,K_in_mmol/L_log_gauss,rock_Andesite,...,rock_Phyllite,rock_Quartzite,rock_Rhyolite,rock_Salt,rock_Sand and Gravel,rock_Sandstone,rock_Schist,rock_Shale,rock_Siltstone,rock_Slate
0,24.450001,0.434981,0.392558,0.608860,0.540239,0.443862,0.476260,-0.536356,-0.143348,0,...,0,0,0,0,0,0,0,0,0,0
1,24.450001,0.017924,0.060933,0.145731,0.254842,0.014391,0.470915,-0.534745,-0.143348,0,...,0,0,0,0,0,1,0,0,0,0
2,24.450001,-1.062569,-1.071109,-0.637923,-0.653693,-1.161474,0.951200,0.402645,-0.143348,0,...,0,0,0,0,0,1,0,0,0,0
3,24.450001,0.308617,0.328322,0.395348,0.407366,0.228954,0.986024,-0.062112,-0.143348,0,...,0,0,0,0,0,1,0,0,0,0
4,24.450001,3.125231,3.109565,3.167914,2.820498,3.084792,-0.034750,-2.591858,-0.143348,0,...,0,0,0,0,0,1,0,0,0,0


<div class="alert alert-info">
    <b>Validation & Documentation</b><br>
    Comparison of Real vs. Synthetic Data distributions to validte the VAE generation quality.
</div>

In [7]:
print("============================================================")
print("       GENERATION STATISTICS                                ")
print("============================================================")

n_real = len(df)
n_synth = len(df_synthetic)

print(f"Real Samples:      {n_real}")
print(f"Synthetic Samples: {n_synth}")
print("-"*60)

# ------------------------- Verteilungsanalyse -------------------------
if 'rock_type' in df.columns and 'rock_type' in df_synthetic.columns:
    plt.figure(figsize=(14, 6))
    
# ------------------------- Gesteinsart-Analyse -------------------------
    real_counts = df['rock_type'].value_counts(normalize=True)
    synth_counts = df_synthetic['rock_type'].value_counts(normalize=True)
    
    all_rocks = sorted(list(set(real_counts.index) | set(synth_counts.index)))
    
    real_vals = [real_counts.get(r, 0) for r in all_rocks]
    synth_vals = [synth_counts.get(r, 0) for r in all_rocks]
    
    x = np.arange(len(all_rocks))
    width = 0.35
    
    plt.bar(x - width/2, real_vals, width, label='Real', color='blue', alpha=0.7)
    plt.bar(x + width/2, synth_vals, width, label='Synthetic', color='orange', alpha=0.7)
    
    plt.xlabel('Rock Type')
    plt.ylabel('Proportion')
    plt.title('Rock Type Distribution Comparison')
    plt.xticks(x, all_rocks, rotation=90)
    plt.legend()
    plt.tight_layout()
    plt.show()


       GENERATION STATISTICS                                
Real Samples:      24332
Synthetic Samples: 24332
------------------------------------------------------------


In [8]:
# ------------------------- SCATTER PLOT REPORT (Real vs Synthetic) -------------------------

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

MAX_POINTS = 3000 # Limit points for performance

print("Analysiere Korrelationen für Scatterplots...")
real_data = df_clean

# FIX: Definieren features_to_plot explicitly based on the dataframe columns
# We filter for numeric columns to be safe
features_to_plot = real_data.select_dtypes(include=[np.number]).columns.tolist()
if not features_to_plot:
    features_to_plot = real_data.columns.tolist() # Fallback

corr_matrix = real_data[features_to_plot].corr()

output_pdf = "VAE_Synthetic_Data_Generation_Report.pdf"
print(f"Erstelle Report: {output_pdf}")

with PdfPages(output_pdf) as pdf:
    plt.figure(figsize=(10, 6))
    plt.text(0.5, 0.5, f"VAE Synthetic Data Validation\n\nContour + Scatter Check\nBlue Lines = Real Data Density\nRed Points = Synthetic Data\n(Points sampled to max {MAX_POINTS})", 
             ha='center', va='center', fontsize=16)
    plt.axis('off')
    pdf.savefig()
    plt.close()

    for i, feature in enumerate(features_to_plot):
        partner = None
        corr_val = 0.0
        try:
            if feature in corr_matrix.columns:
                corrs = corr_matrix[feature].drop(feature, errors='ignore').abs()
                if not corrs.empty:
                    partner = corrs.idxmax()
                    corr_val = corrs.max()
        except Exception:
            pass

        if not partner:
             defaults = ['temperature_in_c', 'Cl_in_mmol/L']
             for d in defaults:
                 if d in real_data.columns and d != feature:
                     partner = d
                     break

        if partner and partner in real_data.columns:
            try:
                fig, ax = plt.subplots(figsize=(10, 8))
                
                # 1. REAL DATA -> KDE CONTOURS (Blue)
                if not real_data.empty:
                     valid_real = real_data[[partner, feature]].dropna()
                     if len(valid_real) > 100:
                         sns.kdeplot(x=partner, y=feature, data=valid_real, 
                                     levels=5, color='blue', alpha=0.7, linewidths=1.5, ax=ax)
                     else:
                         sns.scatterplot(x=partner, y=feature, data=valid_real, 
                                         color='blue', alpha=0.5, ax=ax)
                
                # 2. SYNTHETIC DATA -> SCATTER POINTS (Red)
                if 'synthetic_df' in locals() and not synthetic_df.empty:
                     plot_data_syn = synthetic_df.sample(n=min(len(synthetic_df), MAX_POINTS), random_state=42) if len(synthetic_df) > MAX_POINTS else synthetic_df
                     sns.scatterplot(x=partner, y=feature, data=plot_data_syn, 
                                     color='red', alpha=0.6, label='Synthetic', ax=ax, s=25, marker='x')
                
                ax.set_title(f"Distribution Check: {feature} vs {partner} (Real Corr: {corr_val:.2f})")
                
                from matplotlib.lines import Line2D
                custom_lines = [Line2D([0], [0], color='blue', lw=2),
                                Line2D([0], [0], marker='x', color='w', markeredgecolor='red', markersize=8)]
                ax.legend(custom_lines, ['Real Data (Density)', 'Synthetic Data (Points)'], loc='upper right')
                
                ax.grid(True, alpha=0.3)
                
                pdf.savefig(fig)
                plt.close(fig)
                print(f"Plotted {feature} vs {partner} (Contour)")
            except Exception as e:
                print(f"Error plotting {feature}: {e}")
                plt.close(fig)

    print("PDF Report fertiggestellt.")


Analysiere Korrelationen für Scatterplots...


Erstelle Report: VAE_Synthetic_Data_Generation_Report.pdf


Plotted WGS84_latitude vs Cl_in_mmol/L_log_gauss (Contour)


Plotted WGS84_longitude vs Database_number (Contour)


Plotted Database_number vs WGS84_longitude (Contour)


...

Plotted rock_Sandstone vs rock_Limestone (Contour)


Plotted rock_Schist vs WGS84_longitude (Contour)


Plotted rock_Shale vs rock_Sandstone (Contour)


Plotted rock_Siltstone vs rock_Sandstone (Contour)


Plotted rock_Slate vs WGS84_longitude (Contour)
PDF Report fertiggestellt.


### Conclusion
The synthetic data generator (VAE) has produced a dataset that closely mirrors the statistical properties of the original real-world data. The overlap in distributions indicates that the VAE has successfully learned the manifold of the hydrogeochemical data.